# 🤖 FTFP ML Pipeline - Build & Deploy

**Purpose:** Complete ML pipeline to train models and deploy as Snowflake UDFs  
**Database:** NEW_FTFP  
**Date:** April 2026 (adapted for NEW_FTFP)

This notebook recreates the complete ML infrastructure:
1. Load TRAINING_TBL (40 trucks with labeled telemetry)
2. Apply feature engineering (17 features)
3. Train 3 XGBoost models (classifier + 2 TTF regressors)
4. Save 6 model artifacts to @MODELS
5. Deploy 3 Python UDFs (CLASSIFY_FAILURE_ML, PREDICT_TTF_ML, PREDICT_TTF_TEMPORAL)
6. Create 2 production views (FEATURE_ENGINEERING_VIEW_TEMPORAL, ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF)

**Running this notebook will recreate the exact working ML system currently in production.**


## Step 1: Setup Environment


In [ ]:
# Import packages
import streamlit as st
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
import joblib
import gzip
from io import BytesIO

# ML libraries
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, r2_score

session = get_active_session()
print("✅ Environment ready")


In [ ]:
USE DATABASE NEW_FTFP;
USE SCHEMA DATA;
USE WAREHOUSE NEW_FTFP_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema;


In [ ]:
-- Verify TRAINING_TBL and @MODELS stage
SELECT COUNT(*) as total_rows, COUNT(DISTINCT ENTITY_ID) as trucks FROM TRAINING_TBL;



In [ ]:
CREATE STAGE IF NOT EXISTS ML_MODELS;

## Step 2: Feature Engineering

Apply the EXACT feature engineering used in production (17 features from TRAINING_TBL)


In [ ]:
-- Create training features (matches FEATURE_ENGINEERING_VIEW_TEMPORAL logic)
CREATE OR REPLACE TEMP TABLE TRAINING_FEATURES AS
WITH time_windows AS (
    SELECT
        ENTITY_ID,
        TIMESTAMP,
        TIME_SLICE(TIMESTAMP, 5, 'MINUTE', 'START') as WINDOW_START,
        ENGINE_TEMP,
        TRANS_OIL_PRESSURE,
        BATTERY_VOLTAGE,
        FAILURE_TYPE,
        TIME_TO_FAILURE
    FROM TRAINING_TBL
),
aggregated_features AS (
    SELECT
        ENTITY_ID,
        WINDOW_START,
        MAX(TIMESTAMP) as FEATURE_TIMESTAMP,
        AVG(ENGINE_TEMP) as AVG_ENGINE_TEMP,
        AVG(TRANS_OIL_PRESSURE) as AVG_TRANS_OIL_PRESSURE,
        AVG(BATTERY_VOLTAGE) as AVG_BATTERY_VOLTAGE,
        STDDEV(BATTERY_VOLTAGE) as STDDEV_BATTERY_VOLTAGE,
        STDDEV(ENGINE_TEMP) as STDDEV_ENGINE_TEMP,
        STDDEV(TRANS_OIL_PRESSURE) as STDDEV_TRANS_OIL_PRESSURE,
        MAX(FAILURE_TYPE) as FAILURE_TYPE,
        AVG(TIME_TO_FAILURE) as TIME_TO_FAILURE,
        COUNT(*) as RECORD_COUNT
    FROM time_windows
    GROUP BY ENTITY_ID, WINDOW_START
),
slope_calculations AS (
    SELECT
        a.*,
        (a.AVG_ENGINE_TEMP - LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_ENGINE_TEMP,
        (a.AVG_TRANS_OIL_PRESSURE - LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_TRANS_OIL_PRESSURE,
        (a.AVG_BATTERY_VOLTAGE - LAG(a.AVG_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_BATTERY_VOLTAGE,
        AVG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as ROLLING_AVG_ENGINE_TEMP,
        AVG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as ROLLING_AVG_TRANS_OIL_PRESSURE,
        SUM(a.STDDEV_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as CUMULATIVE_VOLATILITY,
        SUM(CASE WHEN a.STDDEV_BATTERY_VOLTAGE > 0.7 THEN 1 ELSE 0 END) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as ELEVATED_WINDOW_COUNT,
        (a.STDDEV_BATTERY_VOLTAGE - LAG(a.STDDEV_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as VOLATILITY_DELTA,
        (a.AVG_ENGINE_TEMP - LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) -
        (LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP) - LAG(a.AVG_ENGINE_TEMP, 2) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as TEMP_ACCELERATION,
        (a.AVG_TRANS_OIL_PRESSURE - LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) -
        (LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP) - LAG(a.AVG_TRANS_OIL_PRESSURE, 2) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as PRESSURE_ACCELERATION
    FROM aggregated_features a
)
SELECT
    ENTITY_ID,
    FEATURE_TIMESTAMP,
    AVG_ENGINE_TEMP,
    AVG_TRANS_OIL_PRESSURE,
    AVG_BATTERY_VOLTAGE,
    STDDEV_BATTERY_VOLTAGE,
    STDDEV_ENGINE_TEMP,
    STDDEV_TRANS_OIL_PRESSURE,
    COALESCE(SLOPE_ENGINE_TEMP, 0) as SLOPE_ENGINE_TEMP,
    COALESCE(SLOPE_TRANS_OIL_PRESSURE, 0) as SLOPE_TRANS_OIL_PRESSURE,
    COALESCE(SLOPE_BATTERY_VOLTAGE, 0) as SLOPE_BATTERY_VOLTAGE,
    ROLLING_AVG_ENGINE_TEMP,
    ROLLING_AVG_TRANS_OIL_PRESSURE,
    COALESCE(CUMULATIVE_VOLATILITY, 0) as CUMULATIVE_VOLATILITY,
    COALESCE(ELEVATED_WINDOW_COUNT, 0) as ELEVATED_WINDOW_COUNT,
    COALESCE(VOLATILITY_DELTA, 0) as VOLATILITY_DELTA,
    COALESCE(TEMP_ACCELERATION, 0) as TEMP_ACCELERATION,
    COALESCE(PRESSURE_ACCELERATION, 0) as PRESSURE_ACCELERATION,
    FAILURE_TYPE,
    TIME_TO_FAILURE
FROM slope_calculations
WHERE RECORD_COUNT >= 12;

SELECT COUNT(*) as feature_rows FROM TRAINING_FEATURES;


## Step 3: Train Classification Model

Train XGBoost classifier for failure type prediction (NORMAL, ENGINE_FAILURE, TRANSMISSION_FAILURE, ELECTRICAL_FAILURE)


In [ ]:
# Load features
df = session.table("TRAINING_FEATURES").to_pandas()
print(f"Loaded {len(df):,} samples from {df['ENTITY_ID'].nunique()} trucks")

# 11 features for classifier
feature_cols = [
    'AVG_ENGINE_TEMP', 'AVG_TRANS_OIL_PRESSURE', 'AVG_BATTERY_VOLTAGE',
    'STDDEV_BATTERY_VOLTAGE', 'STDDEV_ENGINE_TEMP', 'STDDEV_TRANS_OIL_PRESSURE',
    'SLOPE_ENGINE_TEMP', 'SLOPE_TRANS_OIL_PRESSURE', 'SLOPE_BATTERY_VOLTAGE',
    'ROLLING_AVG_ENGINE_TEMP', 'ROLLING_AVG_TRANS_OIL_PRESSURE'
]

X = df[feature_cols].fillna(0)
y = df['FAILURE_TYPE']

# Label encoding
label_mapping = {'NORMAL': 0, 'ENGINE_FAILURE': 1, 'TRANSMISSION_FAILURE': 2, 'ELECTRICAL_FAILURE': 3}
reverse_mapping = {v: k for k, v in label_mapping.items()}
y_enc = y.map(label_mapping)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

# Train classifier
clf = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, objective='multi:softmax', num_class=4, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"\n✅ Classifier Accuracy: {acc:.4f}")
print(classification_report(y_test, clf.predict(X_test), target_names=list(label_mapping.keys()), digits=3))


In [ ]:
# Save classifier artifacts
print("💾 Saving classifier...")
for obj, name in [(clf, "classifier_v1_0_0.pkl.gz"), 
                   ({'mapping': label_mapping, 'reverse_mapping': reverse_mapping}, "label_mapping_v1_0_0.pkl.gz"),
                   (feature_cols, "feature_columns_v1_0_0.pkl.gz")]:
    buf = BytesIO()
    with gzip.open(buf, 'wb') as f:
        joblib.dump(obj, f)
    buf.seek(0)
    session.file.put_stream(buf, f"@ML_MODELS/models/{name}", auto_compress=False, overwrite=True)
    print(f"✅ {name}")
print("🎉 Classifier saved!")


## Step 4: Train TTF Regression Models

Train 2 models for time-to-failure prediction:
- **PREDICT_TTF_ML**: For ENGINE/TRANSMISSION failures (11 features)
- **PREDICT_TTF_TEMPORAL**: For ELECTRICAL failures (16 features)


In [ ]:
# TTF Model 1: ENGINE + TRANSMISSION (11 features)
df_et = df[df['FAILURE_TYPE'].isin(['ENGINE_FAILURE', 'TRANSMISSION_FAILURE'])].copy()
X_ttf = df_et[feature_cols].fillna(0)
y_ttf = df_et['TIME_TO_FAILURE'].fillna(0)

X_tr, X_te, y_tr, y_te = train_test_split(X_ttf, y_ttf, test_size=0.2, random_state=42)

reg = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
reg.fit(X_tr, y_tr)

mae = mean_absolute_error(y_te, reg.predict(X_te))
print(f"✅ TTF Regressor (Engine/Transmission) MAE: {mae:.2f} hours")

# Save
buf = BytesIO()
with gzip.open(buf, 'wb') as f:
    joblib.dump(reg, f)
buf.seek(0)
session.file.put_stream(buf, "@ML_MODELS/models/regression_v1_0_0.pkl.gz", auto_compress=False, overwrite=True)
print("✅ regression_v1_0_0.pkl.gz saved")


In [ ]:
# TTF Model 2: ELECTRICAL (16 features)
feature_cols_temporal = feature_cols + ['CUMULATIVE_VOLATILITY', 'ELEVATED_WINDOW_COUNT', 'VOLATILITY_DELTA', 'TEMP_ACCELERATION', 'PRESSURE_ACCELERATION']

df_elec = df[df['FAILURE_TYPE'] == 'ELECTRICAL_FAILURE'].copy()
X_temp = df_elec[feature_cols_temporal].fillna(0)
y_temp = df_elec['TIME_TO_FAILURE'].fillna(0)

X_tr_temp, X_te_temp, y_tr_temp, y_te_temp = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

reg_temp = XGBRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42)
reg_temp.fit(X_tr_temp, y_tr_temp)

mae_temp = mean_absolute_error(y_te_temp, reg_temp.predict(X_te_temp))
print(f"✅ TTF Temporal Regressor (Electrical) MAE: {mae_temp:.2f} hours")

# Save model + feature list
for obj, name in [(reg_temp, "regression_temporal_v1_1_0.pkl.gz"), 
                   (feature_cols_temporal, "feature_columns_temporal_v1_1_0.pkl.gz")]:
    buf = BytesIO()
    with gzip.open(buf, 'wb') as f:
        joblib.dump(obj, f)
    buf.seek(0)
    session.file.put_stream(buf, f"@ML_MODELS/models/{name}", auto_compress=False, overwrite=True)
    print(f"✅ {name}")
print("🎉 All 6 model files saved!")


In [ ]:
-- Verify all 6 model files
LIST @ML_MODELS;


## Step 5: Deploy UDFs

Create the 3 Python UDFs using EXACT production definitions


In [ ]:
-- CLASSIFY_FAILURE_ML: Predicts failure type
CREATE OR REPLACE FUNCTION CLASSIFY_FAILURE_ML(
    AVG_ENGINE_TEMP FLOAT, AVG_TRANS_OIL_PRESSURE FLOAT, AVG_BATTERY_VOLTAGE FLOAT,
    STDDEV_BATTERY_VOLTAGE FLOAT, STDDEV_ENGINE_TEMP FLOAT, STDDEV_TRANS_OIL_PRESSURE FLOAT,
    SLOPE_ENGINE_TEMP FLOAT, SLOPE_TRANS_OIL_PRESSURE FLOAT, SLOPE_BATTERY_VOLTAGE FLOAT,
    ROLLING_AVG_ENGINE_TEMP FLOAT, ROLLING_AVG_TRANS_OIL_PRESSURE FLOAT
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('xgboost','numpy','pandas','scikit-learn','joblib')
HANDLER = 'classify'
IMPORTS = ('@ML_MODELS/models/classifier_v1_0_0.pkl.gz', '@ML_MODELS/models/label_mapping_v1_0_0.pkl.gz', '@ML_MODELS/models/feature_columns_v1_0_0.pkl.gz')
COMMENT='ML-based failure classification - Independent'
AS '
import sys
import joblib
import numpy as np

IMPORT_DIRECTORY_NAME = "snowflake_import_directory"
import_dir = sys._xoptions[IMPORT_DIRECTORY_NAME]

clf_model = joblib.load(import_dir + "classifier_v1_0_0.pkl.gz")
label_info = joblib.load(import_dir + "label_mapping_v1_0_0.pkl.gz")

reverse_label_mapping = label_info["reverse_mapping"]

def classify(avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
             stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
             slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
             rolling_avg_engine_temp, rolling_avg_trans_oil_pressure):
    features = np.array([[
        avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
        stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
        slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
        rolling_avg_engine_temp, rolling_avg_trans_oil_pressure
    ]])
    if np.isnan(features).any():
        return "NORMAL"
    prediction = clf_model.predict(features)[0]
    return reverse_label_mapping[int(prediction)]
';


In [ ]:
-- PREDICT_TTF_ML: Predicts hours to failure for ENGINE/TRANSMISSION
CREATE OR REPLACE FUNCTION PREDICT_TTF_ML(
    AVG_ENGINE_TEMP FLOAT, AVG_TRANS_OIL_PRESSURE FLOAT, AVG_BATTERY_VOLTAGE FLOAT,
    STDDEV_BATTERY_VOLTAGE FLOAT, STDDEV_ENGINE_TEMP FLOAT, STDDEV_TRANS_OIL_PRESSURE FLOAT,
    SLOPE_ENGINE_TEMP FLOAT, SLOPE_TRANS_OIL_PRESSURE FLOAT, SLOPE_BATTERY_VOLTAGE FLOAT,
    ROLLING_AVG_ENGINE_TEMP FLOAT, ROLLING_AVG_TRANS_OIL_PRESSURE FLOAT
)
RETURNS FLOAT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('xgboost','numpy','pandas','scikit-learn','joblib')
HANDLER = 'predict_ttf'
IMPORTS = ('@ML_MODELS/models/regression_v1_0_0.pkl.gz')
COMMENT='TTF prediction - basic_11_features'
AS '
import sys
import joblib
import numpy as np

IMPORT_DIRECTORY_NAME = "snowflake_import_directory"
import_dir = sys._xoptions[IMPORT_DIRECTORY_NAME]

reg_model = joblib.load(import_dir + "regression_v1_0_0.pkl.gz")

def predict_ttf(avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
                stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
                slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
                rolling_avg_engine_temp, rolling_avg_trans_oil_pressure):
    features = np.array([[
        avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
        stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
        slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
        rolling_avg_engine_temp, rolling_avg_trans_oil_pressure
    ]])
    if np.isnan(features).any():
        return 0.0
    prediction = reg_model.predict(features)[0]
    return max(0.1, min(24.0, float(prediction)))
';


In [ ]:
-- PREDICT_TTF_TEMPORAL: Predicts hours to failure for ELECTRICAL (16 features)
CREATE OR REPLACE FUNCTION PREDICT_TTF_TEMPORAL(
    AVG_ENGINE_TEMP FLOAT, AVG_TRANS_OIL_PRESSURE FLOAT, AVG_BATTERY_VOLTAGE FLOAT,
    STDDEV_BATTERY_VOLTAGE FLOAT, STDDEV_ENGINE_TEMP FLOAT, STDDEV_TRANS_OIL_PRESSURE FLOAT,
    SLOPE_ENGINE_TEMP FLOAT, SLOPE_TRANS_OIL_PRESSURE FLOAT, SLOPE_BATTERY_VOLTAGE FLOAT,
    ROLLING_AVG_ENGINE_TEMP FLOAT, ROLLING_AVG_TRANS_OIL_PRESSURE FLOAT,
    CUMULATIVE_VOLATILITY FLOAT, ELEVATED_WINDOW_COUNT FLOAT, VOLATILITY_DELTA FLOAT,
    TEMP_ACCELERATION FLOAT, PRESSURE_ACCELERATION FLOAT
)
RETURNS FLOAT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('xgboost','numpy','pandas','scikit-learn','joblib')
HANDLER = 'predict_ttf_temporal'
IMPORTS = ('@ML_MODELS/models/regression_temporal_v1_1_0.pkl.gz')
COMMENT='TTF prediction - temporal_16_features'
AS '
import sys
import joblib
import numpy as np

IMPORT_DIRECTORY_NAME = "snowflake_import_directory"
import_dir = sys._xoptions[IMPORT_DIRECTORY_NAME]

reg_temporal_model = joblib.load(import_dir + "regression_temporal_v1_1_0.pkl.gz")

def predict_ttf_temporal(avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
                        stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
                        slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
                        rolling_avg_engine_temp, rolling_avg_trans_oil_pressure,
                        cumulative_volatility, elevated_window_count, volatility_delta,
                        temp_acceleration, pressure_acceleration):
    features = np.array([[
        avg_engine_temp, avg_trans_oil_pressure, avg_battery_voltage,
        stddev_battery_voltage, stddev_engine_temp, stddev_trans_oil_pressure,
        slope_engine_temp, slope_trans_oil_pressure, slope_battery_voltage,
        rolling_avg_engine_temp, rolling_avg_trans_oil_pressure,
        cumulative_volatility, elevated_window_count, volatility_delta,
        temp_acceleration, pressure_acceleration
    ]])
    if np.isnan(features).any():
        return 0.0
    prediction = reg_temporal_model.predict(features)[0]
    return max(0.1, min(24.0, float(prediction)))
';


In [ ]:
-- Verify UDFs created
SHOW USER FUNCTIONS LIKE '%FAILURE%' IN SCHEMA DATA;


## Step 6: Create Production Views

Deploy FEATURE_ENGINEERING_VIEW_TEMPORAL and ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF


In [ ]:
-- FEATURE_ENGINEERING_VIEW_TEMPORAL: Transforms TELEMETRY into 17 features
CREATE OR REPLACE VIEW FEATURE_ENGINEERING_VIEW_TEMPORAL(
    ENTITY_ID, FEATURE_TIMESTAMP, AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
    STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
    SLOPE_ENGINE_TEMP, SLOPE_TRANS_OIL_PRESSURE, SLOPE_BATTERY_VOLTAGE,
    ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE,
    CUMULATIVE_VOLATILITY, ELEVATED_WINDOW_COUNT, VOLATILITY_DELTA,
    TEMP_ACCELERATION, PRESSURE_ACCELERATION
) AS
WITH time_windows AS (
    SELECT
        ENTITY_ID, TIMESTAMP, TIME_SLICE(TIMESTAMP, 5, 'MINUTE', 'START') as WINDOW_START,
        ENGINE_TEMP, TRANS_OIL_PRESSURE, BATTERY_VOLTAGE
    FROM TELEMETRY
),
aggregated_features AS (
    SELECT
        ENTITY_ID, WINDOW_START, MAX(TIMESTAMP) as FEATURE_TIMESTAMP,
        AVG(ENGINE_TEMP) as AVG_ENGINE_TEMP,
        AVG(TRANS_OIL_PRESSURE) as AVG_TRANS_OIL_PRESSURE,
        AVG(BATTERY_VOLTAGE) as AVG_BATTERY_VOLTAGE,
        STDDEV(BATTERY_VOLTAGE) as STDDEV_BATTERY_VOLTAGE,
        STDDEV(ENGINE_TEMP) as STDDEV_ENGINE_TEMP,
        STDDEV(TRANS_OIL_PRESSURE) as STDDEV_TRANS_OIL_PRESSURE,
        COUNT(*) as RECORD_COUNT
    FROM time_windows
    GROUP BY ENTITY_ID, WINDOW_START
),
slope_calculations AS (
    SELECT a.*,
        (a.AVG_ENGINE_TEMP - LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_ENGINE_TEMP,
        (a.AVG_TRANS_OIL_PRESSURE - LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_TRANS_OIL_PRESSURE,
        (a.AVG_BATTERY_VOLTAGE - LAG(a.AVG_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) / 5.0 as SLOPE_BATTERY_VOLTAGE,
        AVG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as ROLLING_AVG_ENGINE_TEMP,
        AVG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as ROLLING_AVG_TRANS_OIL_PRESSURE,
        SUM(a.STDDEV_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as CUMULATIVE_VOLATILITY,
        SUM(CASE WHEN a.STDDEV_BATTERY_VOLTAGE > 0.7 THEN 1 ELSE 0 END) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as ELEVATED_WINDOW_COUNT,
        (a.STDDEV_BATTERY_VOLTAGE - LAG(a.STDDEV_BATTERY_VOLTAGE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as VOLATILITY_DELTA,
        (a.AVG_ENGINE_TEMP - LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) -
        (LAG(a.AVG_ENGINE_TEMP) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP) - LAG(a.AVG_ENGINE_TEMP, 2) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as TEMP_ACCELERATION,
        (a.AVG_TRANS_OIL_PRESSURE - LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) -
        (LAG(a.AVG_TRANS_OIL_PRESSURE) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP) - LAG(a.AVG_TRANS_OIL_PRESSURE, 2) OVER (PARTITION BY a.ENTITY_ID ORDER BY a.FEATURE_TIMESTAMP)) as PRESSURE_ACCELERATION
    FROM aggregated_features a
)
SELECT
    ENTITY_ID, FEATURE_TIMESTAMP, AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
    STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
    COALESCE(SLOPE_ENGINE_TEMP, 0) as SLOPE_ENGINE_TEMP,
    COALESCE(SLOPE_TRANS_OIL_PRESSURE, 0) as SLOPE_TRANS_OIL_PRESSURE,
    COALESCE(SLOPE_BATTERY_VOLTAGE, 0) as SLOPE_BATTERY_VOLTAGE,
    ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE,
    COALESCE(CUMULATIVE_VOLATILITY, 0) as CUMULATIVE_VOLATILITY,
    COALESCE(ELEVATED_WINDOW_COUNT, 0) as ELEVATED_WINDOW_COUNT,
    COALESCE(VOLATILITY_DELTA, 0) as VOLATILITY_DELTA,
    COALESCE(TEMP_ACCELERATION, 0) as TEMP_ACCELERATION,
    COALESCE(PRESSURE_ACCELERATION, 0) as PRESSURE_ACCELERATION
FROM slope_calculations
WHERE RECORD_COUNT >= 12;


In [ ]:
-- ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF: Combines classification + TTF prediction
CREATE OR REPLACE VIEW ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF(
    PREDICTION_TIMESTAMP, ENTITY_ID, CURRENT_ENGINE_TEMP, CURRENT_TRANS_PRESSURE,
    CURRENT_BATTERY_VOLTAGE, PREDICTED_FAILURE_TYPE, PREDICTED_HOURS_TO_FAILURE, TTF_MODEL_USED
) AS
WITH latest_features AS (
    SELECT
        FEATURE_TIMESTAMP as PREDICTION_TIMESTAMP, ENTITY_ID,
        AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
        STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
        SLOPE_ENGINE_TEMP, SLOPE_TRANS_OIL_PRESSURE, SLOPE_BATTERY_VOLTAGE,
        ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE,
        CUMULATIVE_VOLATILITY, ELEVATED_WINDOW_COUNT, VOLATILITY_DELTA,
        TEMP_ACCELERATION, PRESSURE_ACCELERATION,
        ROW_NUMBER() OVER (PARTITION BY ENTITY_ID ORDER BY FEATURE_TIMESTAMP DESC) as rn
    FROM FEATURE_ENGINEERING_VIEW_TEMPORAL
),
with_classification AS (
    SELECT *,
        CLASSIFY_FAILURE_ML(
            AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
            STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
            SLOPE_ENGINE_TEMP, SLOPE_TRANS_OIL_PRESSURE, SLOPE_BATTERY_VOLTAGE,
            ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE
        ) as PREDICTED_FAILURE_TYPE
    FROM latest_features WHERE rn = 1
)
SELECT
    PREDICTION_TIMESTAMP, ENTITY_ID,
    AVG_ENGINE_TEMP as CURRENT_ENGINE_TEMP,
    AVG_TRANS_OIL_PRESSURE as CURRENT_TRANS_PRESSURE,
    AVG_BATTERY_VOLTAGE as CURRENT_BATTERY_VOLTAGE,
    PREDICTED_FAILURE_TYPE,
    CASE
        WHEN PREDICTED_FAILURE_TYPE = 'ELECTRICAL_FAILURE'
        THEN PREDICT_TTF_TEMPORAL(
            AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
            STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
            SLOPE_ENGINE_TEMP, SLOPE_TRANS_OIL_PRESSURE, SLOPE_BATTERY_VOLTAGE,
            ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE,
            CUMULATIVE_VOLATILITY, ELEVATED_WINDOW_COUNT, VOLATILITY_DELTA,
            TEMP_ACCELERATION, PRESSURE_ACCELERATION
        )
        WHEN PREDICTED_FAILURE_TYPE IN ('ENGINE_FAILURE', 'TRANSMISSION_FAILURE')
        THEN PREDICT_TTF_ML(
            AVG_ENGINE_TEMP, AVG_TRANS_OIL_PRESSURE, AVG_BATTERY_VOLTAGE,
            STDDEV_BATTERY_VOLTAGE, STDDEV_ENGINE_TEMP, STDDEV_TRANS_OIL_PRESSURE,
            SLOPE_ENGINE_TEMP, SLOPE_TRANS_OIL_PRESSURE, SLOPE_BATTERY_VOLTAGE,
            ROLLING_AVG_ENGINE_TEMP, ROLLING_AVG_TRANS_OIL_PRESSURE
        )
        ELSE NULL
    END as PREDICTED_HOURS_TO_FAILURE,
    CASE
        WHEN PREDICTED_FAILURE_TYPE = 'ELECTRICAL_FAILURE' THEN 'temporal_16_features'
        WHEN PREDICTED_FAILURE_TYPE IN ('ENGINE_FAILURE', 'TRANSMISSION_FAILURE') THEN 'basic_11_features'
        ELSE 'none'
    END as TTF_MODEL_USED
FROM with_classification;


In [ ]:
-- Test UDFs
SELECT 
    CLASSIFY_FAILURE_ML(220.0, 45.0, 12.5, 0.3, 5.0, 2.0, 0.5, -0.1, 0.0, 218.0, 44.5) as clf_test,
    PREDICT_TTF_ML(220.0, 45.0, 12.5, 0.3, 5.0, 2.0, 0.5, -0.1, 0.0, 218.0, 44.5) as ttf_test,
    PREDICT_TTF_TEMPORAL(180.0, 45.0, 12.2, 1.2, 3.0, 2.0, 0.1, 0.0, -0.1, 181.0, 45.5, 15.5, 8, 0.3, 0.2, -0.1) as ttf_temp_test;


In [ ]:
-- Test prediction view (will show results when TELEMETRY has data)
SELECT * FROM ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF order by entity_id;


## ✅ Pipeline Complete!

### What Was Created:
1. ✅ Trained 3 XGBoost models from TRAINING_TBL (40 trucks)
2. ✅ Saved 6 model artifacts to @MODELS stage
3. ✅ Deployed 3 UDFs: CLASSIFY_FAILURE_ML, PREDICT_TTF_ML, PREDICT_TTF_TEMPORAL
4. ✅ Created FEATURE_ENGINEERING_VIEW_TEMPORAL
5. ✅ Created ENHANCED_PREDICTIVE_VIEW_HYBRID_TTF

### System is now production-ready!

Start the writer and fast-forward to see automatic predictions appear within 10-15 seconds.
